# Оценка качества OCR-разметки (PP-StructureV3)

Ноутбук для визуальной проверки результатов `indexer_image`.  
Работает с сохранёнными PNG и JSON из `data/ocr_debug/`.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

DATA_DIR = Path("/workspace/data/ocr_debug")

# Выбрать PDF для проверки
pdf_dirs = sorted(d for d in DATA_DIR.iterdir() if d.is_dir())
for d in pdf_dirs:
    n_pages = len(list(d.glob("*.json")))
    print(f"  {d.name}  ({n_pages} pages)")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ── Настройки ──────────────────────────────────────────────
PDF_NAME = pdf_dirs[0].name   # или вписать вручную
DOC_DIR = DATA_DIR / PDF_NAME
MIN_SCORE = 0.15              # порог score для отрисовки боксов
PAGE = 6                      # страница для детального анализа

pages = sorted(DOC_DIR.glob("*.json"))
print(f"{PDF_NAME}: {len(pages)} pages, PAGE={PAGE}, MIN_SCORE={MIN_SCORE}")

In [ ]:
# ── Цвета по типу блока ────────────────────────────────────
LABEL_COLORS = {
    "text":       ("green",  "green"),
    "title":      ("blue",   "blue"),
    "figure":     ("orange", "orange"),
    "table":      ("red",    "red"),
    "header":     ("purple", "purple"),
    "footer":     ("gray",   "gray"),
    "reference":  ("cyan",   "cyan"),
    "equation":   ("magenta","magenta"),
}

def get_color(label: str):
    for key, colors in LABEL_COLORS.items():
        if key in label.lower():
            return colors
    return ("red", "blue")


def plot_page(page_num: int, min_score: float = MIN_SCORE):
    json_path = DOC_DIR / f"page_{page_num:03d}.json"
    img_path  = DOC_DIR / f"page_{page_num:03d}.png"

    if not json_path.exists():
        print(f"page {page_num}: json not found")
        return
    if not img_path.exists():
        print(f"page {page_num}: png not found")
        return

    with open(json_path, encoding="utf-8") as f:
        parsed_items = json.load(f)

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    drawn = 0
    skipped = 0
    for item in parsed_items:
        res = item.get("res", item) if isinstance(item, dict) else item
        if not isinstance(res, dict):
            continue

        boxes = res.get("layout_det_res", {}).get("boxes", [])
        for box in boxes:
            coord = box.get("coordinate")
            label = box.get("label", "")
            score = box.get("score", 0)

            if coord is None or len(coord) != 4:
                continue
            if score < min_score:
                skipped += 1
                continue

            x1, y1, x2, y2 = coord
            outline_color, text_color = get_color(label)
            draw.rectangle([x1, y1, x2, y2], outline=outline_color, width=3)
            draw.text((x1, max(0, y1 - 15)), f"{label} {score:.2f}", fill=text_color)
            drawn += 1

    print(f"Page {page_num}: {drawn} boxes drawn, {skipped} skipped (score < {min_score})")

    plt.figure(figsize=(14, 18))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Page {page_num}")
    plt.show()

In [ ]:
# ── Одна страница ─────────────────────────────────────────
plot_page(PAGE)

In [ ]:
# ── Диапазон страниц ──────────────────────────────────────
for p in range(1, len(pages) + 1):
    plot_page(p)

## Сырой JSON одной страницы

In [ ]:
# ── Посмотреть сырой результат PP-StructureV3 ─────────────
with open(DOC_DIR / f"page_{PAGE:03d}.json", encoding="utf-8") as f:
    raw = json.load(f)

# Структура верхнего уровня
for i, item in enumerate(raw):
    res = item.get("res", item) if isinstance(item, dict) else item
    if isinstance(res, dict):
        print(f"item[{i}] keys: {list(res.keys())}")
        md = res.get("markdown")
        if md:
            print(f"  markdown: {str(md)[:200]}...")
        boxes = res.get("layout_det_res", {}).get("boxes", [])
        if boxes:
            labels = set(b.get("label", "?") for b in boxes)
            print(f"  layout boxes: {len(boxes)}, labels: {labels}")

## Извлечённый текст (как уйдёт в Qdrant)

In [ ]:
# ── Извлечение текста из JSON (без зависимости от fitz/paddle) ──

_SKIP_LABELS = {"header", "footer", "number", "page_number"}

def _extract_text_and_tables(parsed_items):
    """Используем parsing_res_list — блоки с block_order, block_label, block_content."""
    blocks = []
    tables = []

    for item in parsed_items:
        res = item.get("res", item) if isinstance(item, dict) else item
        if not isinstance(res, dict):
            continue

        for t in res.get("table_res_list", []) or []:
            html = t.get("pred_html") or t.get("html")
            if html:
                tables.append(html)

        parsing = res.get("parsing_res_list") or []
        for block in parsing:
            content = block.get("block_content", "").strip()
            if not content:
                continue
            label = block.get("block_label", "")
            if label in _SKIP_LABELS:
                continue
            order = block.get("block_order")
            bbox = block.get("block_bbox", [0, 0, 0, 0])
            if "title" in label:
                content = f"## {content}"
            blocks.append({
                "order": order if order is not None else 9999,
                "y": bbox[1] if len(bbox) >= 2 else 0,
                "label": label,
                "text": content,
            })

        if not parsing:
            ocr = res.get("overall_ocr_res") or res.get("ocr_res") or {}
            rec_texts = ocr.get("rec_texts") or []
            if rec_texts:
                blocks.append({"order": 0, "y": 0, "label": "ocr", "text": "\n".join(rec_texts)})

    blocks.sort(key=lambda b: (b["order"], b["y"]))
    return "\n\n".join(b["text"] for b in blocks).strip(), tables


with open(DOC_DIR / f"page_{PAGE:03d}.json", encoding="utf-8") as f:
    raw = json.load(f)

text, tables = _extract_text_and_tables(raw)
print(f"=== Page {PAGE}: {len(text)} chars, {len(tables)} tables ===\n")
print(text[:3000])
if tables:
    print(f"\n--- Tables ({len(tables)}) ---")
    for i, t in enumerate(tables):
        print(f"\nTable {i+1}:\n{t[:500]}")

## Проверка OCR через LLM

In [ ]:
import requests

OLLAMA_URL = "http://ollama:11434"
MODEL = "qwen3:8b"

def llm_check(text: str, prompt_type: str = "ocr_quality") -> str:
    prompts = {
        "ocr_quality": (
            "Ниже текст, извлечённый OCR из скана технической инструкции. "
            "Проверь качество распознавания:\n"
            "1. Есть ли явные ошибки OCR (перепутанные буквы, бессмысленные слова)?\n"
            "2. Потеряны ли фрагменты текста?\n"
            "3. Правильный ли порядок абзацев?\n"
            "Выведи список найденных проблем. Если текст хороший — напиши 'OK'.\n\n"
            f"ТЕКСТ:\n{text}"
        ),
        "fix": (
            "Ниже текст, извлечённый OCR из скана технической инструкции на русском языке. "
            "Исправь очевидные ошибки OCR, не меняя смысл и структуру. "
            "Верни исправленный текст.\n\n"
            f"ТЕКСТ:\n{text}"
        ),
        "summary": (
            "Кратко опиши содержание этой страницы технической инструкции (2-3 предложения).\n\n"
            f"ТЕКСТ:\n{text}"
        ),
    }

    resp = requests.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompts[prompt_type]}],
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 1024},
    }, timeout=120)
    data = resp.json()
    return data.get("message", {}).get("content", "")

In [ ]:
# ── Проверка качества OCR одной страницы ──────────────────
with open(DOC_DIR / f"page_{PAGE:03d}.json", encoding="utf-8") as f:
    raw = json.load(f)

text, _ = _extract_text_and_tables(raw)
print(f"Page {PAGE}: {len(text)} chars\n")

result = llm_check(text, prompt_type="ocr_quality")
print(result)

In [ ]:
# ── Исправленный текст ────────────────────────────────────
fixed = llm_check(text, prompt_type="fix")
print(fixed)

In [ ]:
# ── Пакетная проверка всех страниц ────────────────────────
problems = []
for p in range(1, len(pages) + 1):
    with open(DOC_DIR / f"page_{p:03d}.json", encoding="utf-8") as f:
        raw = json.load(f)
    text, _ = _extract_text_and_tables(raw)
    if not text:
        problems.append((p, "ПУСТАЯ СТРАНИЦА"))
        continue
    result = llm_check(text, prompt_type="ocr_quality")
    if "ok" not in result.lower()[:10]:
        problems.append((p, result))
    print(f"Page {p:3d}: {'OK' if 'ok' in result.lower()[:10] else 'ISSUES'}")

print(f"\n=== Страниц с проблемами: {len(problems)} ===")
for p, issue in problems:
    print(f"\nPage {p}:\n{issue[:300]}")